# AAE4009 Lab 1 — EEG Cognitive State Classification
**Author:** Rami Chumber (25141517X)

This notebook evaluates multiple classifiers on EEG and physiological signals recorded from 18 pilots during simulated air traffic control tasks. Two datasets and two evaluation schemes are compared:

| | Within-session SSS | Pilot-out CV (LOPO) |
|---|---|---|
| **Frequency-domain** (band power) | Part 1A | Part 1B |
| **Raw time-domain** | Part 2A | Part 2B |

**Within-session SSS** (stratified shuffle split, 5 folds): randomly assigns windows across all pilots to train/test, giving an optimistic upper bound — adjacent overlapping windows may appear in both splits.

**Pilot-out CV** (leave-one-pilot-out): trains on all pilots except one and tests on the held-out pilot. Realistic estimate of cross-subject operational performance.

Pre-computed datasets are loaded directly — no transformation is performed in this notebook. Run `feature_engineering.py` to regenerate the band power features.

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                              recall_score, classification_report)
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
import lightgbm as lgb
import xgboost as xgb

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
print("Imports OK")

In [ ]:
TRANSFORMED_PATH     = "data/large_transformed_train.csv"
RAW_PATH             = "data/train_original.csv"
ANOVA_PATH           = "data/anova_band_features.csv"
RAW_SAMPLE_PER_CLASS = 25_000

EEG_COLS = [
    "eeg_fp1","eeg_f7","eeg_f8","eeg_t4","eeg_t6","eeg_t5","eeg_t3","eeg_fp2",
    "eeg_o1","eeg_p3","eeg_pz","eeg_f3","eeg_fz","eeg_f4","eeg_c4","eeg_p4",
    "eeg_poz","eeg_c3","eeg_cz","eeg_o2",
]
PHYS_COLS       = ["ecg", "r", "gsr"]
ALL_SIGNAL_COLS = EEG_COLS + PHYS_COLS
BANDS           = ["delta", "theta", "alpha", "beta", "gamma"]
N_SSS_SPLITS    = 5
SSS_TEST_SIZE   = 0.3

In [ ]:
def normalise_to_baseline(df, feature_cols,
                           baseline_event="A",
                           group_cols=["pilot", "experiment"],
                           event_col="event"):
    """Z-score each feature relative to that pilot-experiment's resting baseline (event A)."""
    df           = df.copy().reset_index(drop=True)
    feature_cols = [f for f in feature_cols if f in df.columns]
    feat_idx     = [df.columns.get_loc(c) for c in feature_cols]
    for (pilot, exp), group in df.groupby(group_cols):
        row_idx  = group.index.tolist()
        baseline = group.loc[group[event_col] == baseline_event, feature_cols]
        if len(baseline) < 5:
            baseline = group[feature_cols]
        bl_mean = baseline.mean().values
        bl_std  = baseline.std().values
        bl_std  = np.where(~np.isfinite(bl_std) | (bl_std < 1e-8), 1.0, bl_std)
        normalised = (group[feature_cols].values - bl_mean) / bl_std
        normalised = np.where(np.isfinite(normalised), normalised, 0.0)
        df.iloc[row_idx, feat_idx] = normalised
    return df


def make_pipeline(estimator):
    return Pipeline([
        ("scaler",  StandardScaler()),
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("model",   estimator),
    ])


def build_feature_sets(anova_df, all_feature_cols):
    return {
        "liberal":    [f for f in anova_df.loc[anova_df["eta2"] > 0.005, "feature"] if f in all_feature_cols],
        "medium":     [f for f in anova_df.loc[anova_df["eta2"] > 0.01,  "feature"] if f in all_feature_cols],
        "strict":     [f for f in anova_df.loc[anova_df["eta2"] > 0.02,  "feature"] if f in all_feature_cols],
        "beta_gamma": [f for f in all_feature_cols
                       if (f.endswith("_beta") or f.endswith("_gamma"))
                       or any(f.startswith(p + "_") or f == p for p in PHYS_COLS)],
    }


def run_sss(df_norm, models, all_feature_cols, y_all, label_name="SSS"):
    """5-fold stratified shuffle split on a pre-normalised DataFrame."""
    sss     = StratifiedShuffleSplit(n_splits=N_SSS_SPLITS, test_size=SSS_TEST_SIZE, random_state=42)
    X_split = df_norm[all_feature_cols].values
    summary = []
    print(f"  {'Model':<24}  {'Acc':>6}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}  {'Std':>5}")
    print("  " + "-" * 57)
    for name, model, feat_cols in models:
        feat_cols = [f for f in feat_cols if f in df_norm.columns]
        X         = df_norm[feat_cols].values
        accs, precs, recs, f1s = [], [], [], []
        for train_idx, test_idx in sss.split(X_split, y_all):
            m = clone(model)
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y_all[train_idx], y_all[test_idx]
            if "XGBoost" in name:
                m.fit(X_tr, y_tr, sample_weight=compute_sample_weight("balanced", y_tr))
            else:
                m.fit(X_tr, y_tr)
            y_pred = m.predict(X_te)
            accs.append(accuracy_score(y_te, y_pred))
            precs.append(precision_score(y_te, y_pred, average="macro", zero_division=0))
            recs.append(recall_score(y_te, y_pred,    average="macro", zero_division=0))
            f1s.append(f1_score(y_te, y_pred,         average="macro", zero_division=0))
        row = {"Model": name, "Eval": label_name,
               "Acc": np.mean(accs), "Prec": np.mean(precs),
               "Rec": np.mean(recs), "F1":   np.mean(f1s), "Std": np.std(f1s)}
        summary.append(row)
        print(f"  {name:<24}  {row['Acc']:>6.3f}  {row['Prec']:>6.3f}  "
              f"{row['Rec']:>6.3f}  {row['F1']:>6.3f}  {row['Std']:>5.3f}")
    return pd.DataFrame(summary).sort_values("F1", ascending=False)


def run_lopo(band_df, models, all_feature_cols, label_map, class_names, label_name="LOPO"):
    """Leave-one-pilot-out CV with per-fold independent baseline normalisation."""
    pilots  = sorted(band_df["pilot"].unique())
    results = {name: [] for name, _, _ in models}
    for held_out in pilots:
        fold_start = time.time()
        print(f"  Fold: held-out pilot {held_out}  ({pilots.index(held_out)+1}/{len(pilots)})")
        train_raw = band_df[band_df["pilot"] != held_out]
        test_raw  = band_df[band_df["pilot"] == held_out]
        train_df  = normalise_to_baseline(train_raw, all_feature_cols)
        test_df   = normalise_to_baseline(test_raw,  all_feature_cols)
        train_df["label"] = train_raw["event"].map(label_map).values
        test_df["label"]  = test_raw["event"].map(label_map).values
        y_train = train_df["label"].values
        y_test  = test_df["label"].values
        for name, model, feat_cols in models:
            feat_cols = [f for f in feat_cols if f in band_df.columns]
            m = clone(model)
            X_tr, X_te = train_df[feat_cols].values, test_df[feat_cols].values
            if "XGBoost" in name:
                m.fit(X_tr, y_train, sample_weight=compute_sample_weight("balanced", y_train))
            else:
                m.fit(X_tr, y_train)
            y_pred   = m.predict(X_te)
            macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
            results[name].append({
                "pilot":    held_out,
                "macro_f1": macro_f1,
                "report":   classification_report(y_test, y_pred, target_names=class_names,
                                                  output_dict=True, zero_division=0),
            })
            print(f"    {name:<24} F1={macro_f1:.3f}")
        print(f"    fold time: {time.time()-fold_start:.0f}s")
    print(f"\n  {'Model':<24}  {'Mean F1':>8}  {'Std':>6}  {'Min':>6}  {'Max':>6}")
    print("  " + "-" * 55)
    summary = []
    for name, _, _ in models:
        f1s = [r["macro_f1"] for r in results[name]]
        row = {"Model": name, "Eval": label_name,
               "F1": np.mean(f1s), "Std": np.std(f1s),
               "Min": np.min(f1s), "Max": np.max(f1s)}
        summary.append(row)
        print(f"  {name:<24}  {row['F1']:>8.3f}  {row['Std']:>6.3f}  "
              f"{row['Min']:>6.3f}  {row['Max']:>6.3f}")
    return pd.DataFrame(summary).sort_values("F1", ascending=False), results, pilots


def plot_lopo_folds(results, model_names, pilots, title, save_path, top_n=5):
    top_models = sorted(model_names,
                        key=lambda n: np.mean([r["macro_f1"] for r in results[n]]),
                        reverse=True)[:top_n]
    fig, ax = plt.subplots(figsize=(12, 5))
    for name in top_models:
        f1s = [r["macro_f1"] for r in results[name]]
        ax.plot(range(len(pilots)), f1s, marker="o", label=name, linewidth=1.5)
    ax.set_xticks(range(len(pilots)))
    ax.set_xticklabels([f"Pilot {p}" for p in pilots], rotation=30, ha="right")
    ax.set_ylabel("Macro F1")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


print("Helper functions defined.")

## Part 1 — Frequency-Domain Classification

Band power features are loaded from `data/large_transformed_train.csv` (pre-computed by `feature_engineering.py` using a sliding-window Welch PSD). Feature sets are filtered by ANOVA η² from `data/anova_band_features.csv`.

Baseline normalisation (z-score relative to each pilot-experiment's event-A windows) is applied before training to reduce between-pilot absolute EEG amplitude differences.

In [ ]:
print("Loading transformed dataset (18-pilot)...")
band_df  = pd.read_csv(TRANSFORMED_PATH)
anova_df = pd.read_csv(ANOVA_PATH)
print(f"  Rows: {len(band_df):,}  |  Pilots: {band_df['pilot'].nunique()}  |  Classes: {sorted(band_df['event'].unique())}")

eeg_band_cols = [c for c in band_df.columns
                 if c.startswith("eeg_") and any(c.endswith(f"_{b}") for b in BANDS)]
phys_feat_cols = [c for c in band_df.columns
                  if any(c.startswith(p + "_") or c == p for p in PHYS_COLS)]
all_freq_cols  = eeg_band_cols + phys_feat_cols
zero_var       = [f for f in all_freq_cols if band_df[f].std() == 0]
all_freq_cols  = [f for f in all_freq_cols if f not in zero_var]
if zero_var:
    print(f"  Dropped {len(zero_var)} zero-variance features")

feat = build_feature_sets(anova_df, all_freq_cols)
print(f"  Features — total: {len(all_freq_cols)}  liberal: {len(feat['liberal'])}  "
      f"strict: {len(feat['strict'])}  \u03b2/\u03b3: {len(feat['beta_gamma'])}")

label_map_freq     = {label: i for i, label in enumerate(sorted(band_df["event"].unique()))}
inv_label_map_freq = {v: k for k, v in label_map_freq.items()}
band_df["label"]   = band_df["event"].map(label_map_freq)
class_names_freq   = [inv_label_map_freq[i] for i in range(len(label_map_freq))]
print(f"  Label mapping: {label_map_freq}")
print("  Windows per class:")
print(band_df["event"].value_counts().sort_index().to_string())

### 1A — Within-Session Stratified Shuffle Split

5-fold SSS (`test_size=0.3`). All 18 pilots' windows are pooled before splitting, so adjacent windows from the same pilot can appear in both train and test. **Scores are optimistic** — treat as an upper bound. Pilot-out CV (1B) is the more reliable evaluation.

Baseline normalisation is applied to the full pooled dataset before splitting (no pilot leakage — each pilot's normalisation uses only their own event-A windows).

In [ ]:
print("Applying baseline normalisation to pooled dataset...")
band_norm          = normalise_to_baseline(band_df, all_freq_cols)
band_norm["label"] = band_df["label"].values
y_freq             = band_norm["label"].values

SSS_FREQ_MODELS = [
    ("LightGBM",
     lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                        num_leaves=31, min_child_samples=20, colsample_bytree=0.8,
                        subsample=0.8, class_weight="balanced", random_state=42, verbose=-1),
     feat["liberal"]),
    ("XGBoost",
     xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                       colsample_bytree=0.8, subsample=0.8, eval_metric="mlogloss",
                       random_state=42, verbosity=0),
     feat["liberal"]),
    ("Random Forest",
     RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5,
                            max_features="sqrt", class_weight="balanced",
                            random_state=42, n_jobs=-1),
     feat["liberal"]),
    ("SVM (Linear)",
     make_pipeline(LinearSVC(C=1.0, class_weight="balanced", max_iter=2000, random_state=42)),
     feat["strict"]),
    ("KNN",
     make_pipeline(KNeighborsClassifier(n_neighbors=11, weights="distance",
                                        metric="euclidean", n_jobs=-1)),
     feat["strict"]),
    ("MLP",
     make_pipeline(MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", alpha=0.01,
                                 learning_rate="adaptive", max_iter=200,
                                 early_stopping=True, validation_fraction=0.1, random_state=42)),
     feat["strict"]),
]

print("\nFrequency-domain within-session SSS (5 folds):")
sss_freq_df = run_sss(band_norm, SSS_FREQ_MODELS, all_freq_cols, y_freq, label_name="Freq SSS")
sss_freq_df.to_csv("results/freq_sss_results.csv", index=False)
print(f"\nSaved: results/freq_sss_results.csv")

### 1B — Pilot-Out Cross-Validation (Frequency-Domain)

Leave-one-pilot-out CV across all 18 pilots. Baseline normalisation is applied **independently** to the training and test partitions in each fold — preventing cross-pilot information leakage.

Additional β/γ-only feature sets are evaluated here. Beta and gamma bands show lower between-pilot coefficient of variation than theta/alpha bands (shown in `eda.py`), so these variants test whether restricting to higher-frequency features improves cross-subject generalisation.

In [ ]:
LOPO_FREQ_MODELS = [
    ("LightGBM",
     lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                        num_leaves=31, min_child_samples=20, colsample_bytree=0.8,
                        subsample=0.8, class_weight="balanced", random_state=42, verbose=-1),
     feat["liberal"]),
    ("LightGBM (\u03b2/\u03b3)",
     lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                        num_leaves=31, min_child_samples=20, colsample_bytree=0.8,
                        subsample=0.8, class_weight="balanced", random_state=42, verbose=-1),
     feat["beta_gamma"]),
    ("XGBoost",
     xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                       colsample_bytree=0.8, subsample=0.8, eval_metric="mlogloss",
                       random_state=42, verbosity=0),
     feat["liberal"]),
    ("XGBoost (\u03b2/\u03b3)",
     xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                       colsample_bytree=0.8, subsample=0.8, eval_metric="mlogloss",
                       random_state=42, verbosity=0),
     feat["beta_gamma"]),
    ("Random Forest",
     RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5,
                            max_features="sqrt", class_weight="balanced",
                            random_state=42, n_jobs=-1),
     feat["liberal"]),
    ("SVM (Linear)",
     Pipeline([("scaler", StandardScaler()),
               ("svm", LinearSVC(C=1.0, class_weight="balanced", max_iter=2000, random_state=42))]),
     feat["strict"]),
    ("SVM (Linear \u03b2/\u03b3)",
     Pipeline([("scaler", StandardScaler()),
               ("svm", LinearSVC(C=1.0, class_weight="balanced", max_iter=2000, random_state=42))]),
     feat["beta_gamma"]),
    ("KNN",
     Pipeline([("scaler", StandardScaler()),
               ("knn", KNeighborsClassifier(n_neighbors=11, weights="distance",
                                            metric="euclidean", n_jobs=-1))]),
     feat["strict"]),
    ("MLP",
     Pipeline([("scaler", StandardScaler()),
               ("mlp", MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", alpha=0.01,
                                     learning_rate="adaptive", max_iter=200,
                                     early_stopping=True, validation_fraction=0.1, random_state=42))]),
     feat["medium"]),
]

print("Frequency-domain pilot-out CV (leave-one-pilot-out):")
lopo_freq_df, lopo_freq_results, freq_pilots = run_lopo(
    band_df, LOPO_FREQ_MODELS, all_freq_cols, label_map_freq, class_names_freq,
    label_name="Freq LOPO"
)
lopo_freq_df.to_csv("results/freq_lopo_results.csv", index=False)
print(f"\nSaved: results/freq_lopo_results.csv")

freq_model_names = [name for name, _, _ in LOPO_FREQ_MODELS]
plot_lopo_folds(lopo_freq_results, freq_model_names, freq_pilots,
                title="Pilot-out CV — macro F1 per fold (frequency-domain, top 5 models)",
                save_path="figures/freq_lopo_folds.png")

## Part 2 — Raw Time-Domain Classification

Raw EEG and physiological signals are used directly, without frequency-domain transformation. This serves as a baseline to quantify how much the Welch band power features improve classification performance.

Per-pilot MinMax normalisation is applied within each evaluation fold to prevent cross-pilot leakage. The raw dataset is subsampled to `RAW_SAMPLE_PER_CLASS` rows per class to keep training tractable.

In [ ]:
print("Loading raw dataset (18-pilot)...")
raw_df_full = pd.read_csv(RAW_PATH)
raw_df_full["pilot"] = 100 * raw_df_full["seat"] + raw_df_full["crew"]
print(f"  Rows: {len(raw_df_full):,}  |  Pilots: {raw_df_full['pilot'].nunique()}")
print(f"  Classes: {sorted(raw_df_full['event'].unique())}")
print(f"  Experiments: {sorted(raw_df_full['experiment'].unique())}")

label_map_raw     = {label: i for i, label in enumerate(sorted(raw_df_full["event"].unique()))}
inv_label_map_raw = {v: k for k, v in label_map_raw.items()}
class_names_raw   = [inv_label_map_raw[i] for i in range(len(label_map_raw))]
print(f"  Label mapping: {label_map_raw}")

### 2A — Within-Session SSS (Raw Time-Domain)

Same 5-fold SSS scheme as 1A, applied to the subsampled raw signals. Per-pilot MinMax normalisation is applied to the subsampled dataset before splitting.

No β/γ variants are included for raw data — band-specific feature selection is a frequency-domain concept.

In [ ]:
print(f"Subsampling raw data ({RAW_SAMPLE_PER_CLASS:,} rows per class max)...")
raw_sss = (raw_df_full.groupby("event", group_keys=False)
                      .apply(lambda x: x.sample(min(len(x), RAW_SAMPLE_PER_CLASS), random_state=42))
                      .reset_index(drop=True))
print(f"  Subsampled: {len(raw_sss):,} rows")
print(raw_sss["event"].value_counts().sort_index().to_string())

print("\nApplying per-pilot MinMax normalisation...")
raw_sss_norm = raw_sss.copy()
for pilot, idx in raw_sss_norm.groupby("pilot").groups.items():
    scaler = MinMaxScaler()
    raw_sss_norm.loc[idx, ALL_SIGNAL_COLS] = scaler.fit_transform(
        raw_sss_norm.loc[idx, ALL_SIGNAL_COLS]
    )
raw_sss_norm["label"] = raw_sss_norm["event"].map(label_map_raw)
y_raw_sss = raw_sss_norm["label"].values

RAW_SSS_MODELS = [
    ("LightGBM",
     lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                        num_leaves=31, min_child_samples=20, colsample_bytree=0.8,
                        subsample=0.8, class_weight="balanced", random_state=42, verbose=-1),
     ALL_SIGNAL_COLS),
    ("XGBoost",
     xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                       colsample_bytree=0.8, subsample=0.8, eval_metric="mlogloss",
                       random_state=42, verbosity=0),
     ALL_SIGNAL_COLS),
    ("Random Forest",
     RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5,
                            max_features="sqrt", class_weight="balanced",
                            random_state=42, n_jobs=-1),
     ALL_SIGNAL_COLS),
    ("SVM (Linear)",
     make_pipeline(LinearSVC(C=1.0, class_weight="balanced", max_iter=2000, random_state=42)),
     ALL_SIGNAL_COLS),
    ("KNN",
     make_pipeline(KNeighborsClassifier(n_neighbors=11, weights="distance",
                                        metric="euclidean", n_jobs=-1)),
     ALL_SIGNAL_COLS),
    ("MLP",
     make_pipeline(MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", alpha=0.01,
                                 learning_rate="adaptive", max_iter=200,
                                 early_stopping=True, validation_fraction=0.1, random_state=42)),
     ALL_SIGNAL_COLS),
]

print("\nRaw time-domain within-session SSS (5 folds):")
sss_raw_df = run_sss(raw_sss_norm, RAW_SSS_MODELS, ALL_SIGNAL_COLS, y_raw_sss, label_name="Raw SSS")
sss_raw_df.to_csv("results/raw_sss_results.csv", index=False)
print(f"\nSaved: results/raw_sss_results.csv")

### 2B — Pilot-Out CV (Raw Time-Domain)

Leave-one-pilot-out CV on raw signals. Within each fold, per-pilot MinMax normalisation is applied **independently** to the training pilots and the held-out test pilot to prevent leakage. The training set is subsampled to `RAW_SAMPLE_PER_CLASS` rows per class; the test pilot's full data is used for evaluation.

In [ ]:
def run_raw_lopo(raw_df_full, models, feature_cols, label_map, class_names):
    """LOPO CV for raw time-domain data with per-fold per-pilot MinMax normalisation."""
    pilots  = sorted(raw_df_full["pilot"].unique())
    results = {name: [] for name, _, _ in models}
    for held_out in pilots:
        fold_start = time.time()
        print(f"  Fold: held-out pilot {held_out}  ({pilots.index(held_out)+1}/{len(pilots)})")
        train_pool = raw_df_full[raw_df_full["pilot"] != held_out].copy()
        test_raw   = raw_df_full[raw_df_full["pilot"] == held_out].copy()
        # Subsample training data per class
        train_sub = (train_pool.groupby("event", group_keys=False)
                               .apply(lambda x: x.sample(min(len(x), RAW_SAMPLE_PER_CLASS), random_state=42))
                               .reset_index(drop=True))
        # Per-pilot MinMax — applied independently to training and test partitions
        for pilot, idx in train_sub.groupby("pilot").groups.items():
            sc = MinMaxScaler()
            train_sub.loc[idx, feature_cols] = sc.fit_transform(train_sub.loc[idx, feature_cols])
        sc = MinMaxScaler()
        test_raw[feature_cols] = sc.fit_transform(test_raw[feature_cols])
        train_sub["label"] = train_sub["event"].map(label_map)
        test_raw["label"]  = test_raw["event"].map(label_map)
        y_train = train_sub["label"].values
        y_test  = test_raw["label"].values
        for name, model, feat_cols in models:
            feat_cols = [f for f in feat_cols if f in train_sub.columns]
            m = clone(model)
            X_tr, X_te = train_sub[feat_cols].values, test_raw[feat_cols].values
            if "XGBoost" in name:
                m.fit(X_tr, y_train, sample_weight=compute_sample_weight("balanced", y_train))
            else:
                m.fit(X_tr, y_train)
            y_pred   = m.predict(X_te)
            macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
            results[name].append({
                "pilot":    held_out,
                "macro_f1": macro_f1,
                "report":   classification_report(y_test, y_pred, target_names=class_names,
                                                  output_dict=True, zero_division=0),
            })
            print(f"    {name:<24} F1={macro_f1:.3f}")
        print(f"    fold time: {time.time()-fold_start:.0f}s")
    print(f"\n  {'Model':<24}  {'Mean F1':>8}  {'Std':>6}  {'Min':>6}  {'Max':>6}")
    print("  " + "-" * 55)
    summary = []
    for name, _, _ in models:
        f1s = [r["macro_f1"] for r in results[name]]
        row = {"Model": name, "Eval": "Raw LOPO",
               "F1": np.mean(f1s), "Std": np.std(f1s),
               "Min": np.min(f1s), "Max": np.max(f1s)}
        summary.append(row)
        print(f"  {name:<24}  {row['F1']:>8.3f}  {row['Std']:>6.3f}  "
              f"{row['Min']:>6.3f}  {row['Max']:>6.3f}")
    return pd.DataFrame(summary).sort_values("F1", ascending=False), results, pilots


LOPO_RAW_MODELS = [
    ("LightGBM",
     lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                        num_leaves=31, min_child_samples=20, colsample_bytree=0.8,
                        subsample=0.8, class_weight="balanced", random_state=42, verbose=-1),
     ALL_SIGNAL_COLS),
    ("XGBoost",
     xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                       colsample_bytree=0.8, subsample=0.8, eval_metric="mlogloss",
                       random_state=42, verbosity=0),
     ALL_SIGNAL_COLS),
    ("Random Forest",
     RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5,
                            max_features="sqrt", class_weight="balanced",
                            random_state=42, n_jobs=-1),
     ALL_SIGNAL_COLS),
    ("SVM (Linear)",
     Pipeline([("scaler", StandardScaler()),
               ("svm", LinearSVC(C=1.0, class_weight="balanced", max_iter=2000, random_state=42))]),
     ALL_SIGNAL_COLS),
    ("KNN",
     Pipeline([("scaler", StandardScaler()),
               ("knn", KNeighborsClassifier(n_neighbors=11, weights="distance",
                                            metric="euclidean", n_jobs=-1))]),
     ALL_SIGNAL_COLS),
    ("MLP",
     Pipeline([("scaler", StandardScaler()),
               ("mlp", MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", alpha=0.01,
                                     learning_rate="adaptive", max_iter=200,
                                     early_stopping=True, validation_fraction=0.1, random_state=42))]),
     ALL_SIGNAL_COLS),
]

print("Raw time-domain pilot-out CV (leave-one-pilot-out):")
raw_lopo_df, raw_lopo_results, raw_pilots = run_raw_lopo(
    raw_df_full, LOPO_RAW_MODELS, ALL_SIGNAL_COLS, label_map_raw, class_names_raw
)
raw_lopo_df.to_csv("results/raw_lopo_results.csv", index=False)
print(f"\nSaved: results/raw_lopo_results.csv")

raw_model_names = [name for name, _, _ in LOPO_RAW_MODELS]
plot_lopo_folds(raw_lopo_results, raw_model_names, raw_pilots,
                title="Pilot-out CV \u2014 macro F1 per fold (raw time-domain, top 5 models)",
                save_path="figures/raw_lopo_folds.png")

## Part 3 — Comparison Summary

Macro F1 across all four conditions for the six base classifiers.

| Condition | Notes |
|---|---|
| **Freq SSS** | Band power features, within-session (optimistic) |
| **Freq LOPO** | Band power features, pilot-out (realistic) |
| **Raw SSS** | Raw signals, within-session (optimistic) |
| **Raw LOPO** | Raw signals, pilot-out (realistic) |

The gap between SSS and LOPO reflects how much within-session autocorrelation inflates scores. The gap between Freq and Raw reflects the benefit of the frequency-domain feature extraction.

In [ ]:
BASE_MODELS = ["LightGBM", "XGBoost", "Random Forest", "SVM (Linear)", "KNN", "MLP"]

eval_frames = {
    "Freq SSS":  sss_freq_df.set_index("Model")["F1"],
    "Freq LOPO": lopo_freq_df.set_index("Model")["F1"],
    "Raw SSS":   sss_raw_df.set_index("Model")["F1"],
    "Raw LOPO":  raw_lopo_df.set_index("Model")["F1"],
}

comparison = pd.DataFrame({
    condition: {m: scores.get(m, np.nan) for m in BASE_MODELS}
    for condition, scores in eval_frames.items()
})
comparison.index.name = "Model"
comparison = comparison.round(3)

comparison.to_csv("results/comparison_summary.csv")
print("Comparison table (macro F1):")
print(comparison.to_string())
print(f"\nSaved: results/comparison_summary.csv")

conditions = list(eval_frames.keys())
x          = np.arange(len(BASE_MODELS))
width      = 0.2
colors     = ["#1D9E75", "#5B5EA6", "#FFC857", "#E05C5C"]
offsets    = [-1.5, -0.5, 0.5, 1.5]

fig, ax = plt.subplots(figsize=(13, 5))
for cond, color, offset in zip(conditions, colors, offsets):
    vals = [comparison.loc[m, cond] if m in comparison.index else np.nan for m in BASE_MODELS]
    ax.bar(x + offset * width, vals, width, label=cond, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(BASE_MODELS, rotation=15, ha="right")
ax.set_ylabel("Macro F1")
ax.set_title("Raw vs Frequency-Domain \u2014 Within-Session SSS vs Pilot-Out CV")
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("figures/comparison_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/comparison_summary.png")